# Notebook 21d: SH Level Scaling for Regional Tasks

**Date**: 2026-01-12  
**Goal**: Test if L=40 outperforms L=10 for smaller regional tasks

**Hypothesis**: SH level should match spatial scale
- Global tasks: L=10 sufficient (low frequencies dominate)
- Regional tasks: L=40 captures higher frequencies better

---

## Motivation

**User observation**:
> "With smaller coverage size, L=40 was able to be trained to be better than L=10 in some cases, mainly if the region being tested is smaller."

**Makes sense theoretically**:
- L=40 (1681 dims) captures higher spatial frequencies than L=10 (121 dims)
- Smaller regions have higher relative frequency content
- Trade-off: L=40 adds 1560 parameters vs L=10

**This experiment tests**:
1. Does L=40 improve R² for regional tasks?
2. Is improvement worth 13× parameter increase?
3. Does activation choice interact with L-value?

---

## Experimental Design

### Configs Tested
- **Regions**: Himalayas (complex), Sahara (flat)
- **L-values**: L=10 (121 dims), L=40 (1681 dims)
- **Activations**: ReLU, Spline
- **Seeds**: 5 per configuration
- **Samples**: 20K (larger regional sample)

### Expected Outcomes

**Scenario A: L=40 wins on complex terrain**
```
Himalayas (mountainous):
  L=10+ReLU:  R² = 0.75
  L=40+ReLU:  R² = 0.82  (+9% improvement)
  → L=40 captures high-freq terrain features

Sahara (flat):
  L=10+ReLU:  R² = 0.88
  L=40+ReLU:  R² = 0.88  (no improvement)
  → Low-freq sufficient for flat terrain
```

**Scenario B: L=40 always helps regionally**
```
Both regions show +5-10% with L=40
→ Regional tasks need higher frequencies
→ Justifies parameter cost
```

**Scenario C: L=40 doesn't help**
```
L=10 ≈ L=40 for both regions
→ L=10 already captures sufficient detail
→ L=40 just adds parameters without benefit
```

---

## Computational Budget

**Total**: ~2.5 hours on Colab T4

**Breakdown**:
- 2 regions × 2 L-values × 5 seeds × 2 activations × 100s ≈ 2.5 hours

**Much faster than full NB21 regional** because:
- Only 5 seeds (not 10)
- Only 20K samples (not 10K+20K)
- Focused comparison

---

## Integration with NB21

**Compare to NB21 Exp 3** (if it ran L=10 regional):
- NB21: L=10 regional results with 5-10 seeds
- NB21d: L=40 regional results with 5 seeds
- Direct comparison shows L scaling effect

**If NB21 already tested L=40**: Skip this notebook, redundant.

**If NB21 skipped regional**: This provides regional baseline.

---
## Setup

In [ ]:
# Environment setup
import os
import sys

if 'COLAB_GPU' in os.environ:
    !rm -rf sample_data .config satclip 2>/dev/null
    !git clone https://github.com/1hamzaiqbal/satclip.git
    !pip install lightning torchgeo huggingface_hub rasterio --quiet
    sys.path.append('./satclip/satclip')
else:
    sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'satclip'))

In [ ]:
# Imports
import numpy as np
import pandas as pd
import time
import warnings
from scipy import stats
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import r2_score

import positional_encoding as PE
import xarray as xr

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

---
## Model Definitions (Same as NB21)

Using same architecture, just varying `sh_legendre_polys` (L-value)

In [ ]:
# Copy SplineActivation, SirenLayer, UniversalEncoder, RegressionPredictor from NB21
# (Same classes, not repeated here for brevity)

class SplineActivation(nn.Module):
    def __init__(self, n_knots=15, input_range=(-3.0, 3.0), init='relu'):
        super().__init__()
        self.n_knots = n_knots
        self.input_range = input_range
        knot_x = torch.linspace(input_range[0], input_range[1], n_knots)
        self.register_buffer('knot_x', knot_x)
        if init == 'relu':
            knot_y = torch.relu(knot_x)
        elif init == 'linear':
            knot_y = knot_x.clone()
        else:
            knot_y = torch.randn(n_knots) * 0.1
        self.knot_y = nn.Parameter(knot_y)

    def forward(self, x):
        x_clamped = torch.clamp(x, self.input_range[0], self.input_range[1])
        x_norm = (x_clamped - self.knot_x[0]) / (self.knot_x[-1] - self.knot_x[0])
        x_idx = x_norm * (self.n_knots - 1)
        idx_low = torch.floor(x_idx).long()
        idx_high = torch.clamp(idx_low + 1, max=self.n_knots - 1)
        idx_low = torch.clamp(idx_low, max=self.n_knots - 1)
        weight = x_idx - idx_low.float()
        y_low = self.knot_y[idx_low]
        y_high = self.knot_y[idx_high]
        return y_low + weight * (y_high - y_low)


class UniversalEncoder(nn.Module):
    def __init__(self, input_type='sh', sh_legendre_polys=10,
                 activation_type='spline', activation_kwargs=None,
                 n_layers=3, hidden_dim=256, output_dim=256):
        super().__init__()
        self.input_type = input_type
        self.activation_type = activation_type

        if input_type == 'sh':
            self.posenc = PE.SphericalHarmonics(
                legendre_polys=sh_legendre_polys,
                harmonics_calculation='analytic'
            )
            with torch.no_grad():
                test_coords = torch.zeros(1, 2)
                test_output = self.posenc(test_coords)
                input_dim = test_output.shape[1]
        else:
            raise ValueError(f"Unknown input_type: {input_type}")

        if activation_kwargs is None:
            activation_kwargs = {}

        dims = [input_dim] + [hidden_dim] * n_layers + [output_dim]

        if activation_type == 'spline':
            self.linears = nn.ModuleList([
                nn.Linear(dims[i], dims[i+1])
                for i in range(len(dims) - 1)
            ])
            self.activations = nn.ModuleList([
                SplineActivation(**activation_kwargs)
                for _ in range(n_layers)
            ])
            for linear in self.linears:
                nn.init.kaiming_normal_(linear.weight)
                nn.init.zeros_(linear.bias)

        elif activation_type == 'relu':
            layers = []
            for i in range(len(dims) - 1):
                layers.append(nn.Linear(dims[i], dims[i+1]))
                if i < len(dims) - 2:
                    layers.append(nn.ReLU())
            self.net = nn.Sequential(*layers)
            for m in self.modules():
                if isinstance(m, nn.Linear):
                    nn.init.kaiming_normal_(m.weight)
                    nn.init.zeros_(m.bias)
        else:
            raise ValueError(f"Unknown activation_type: {activation_type}")

    def forward(self, coords):
        x = self.posenc(coords)

        if self.activation_type == 'spline':
            for i, (linear, act) in enumerate(zip(self.linears[:-1], self.activations)):
                x = act(linear(x))
            x = self.linears[-1](x)
        else:
            x = self.net(x)

        return x


class RegressionPredictor(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 1)
        )

    def forward(self, coords):
        return self.head(self.encoder(coords)).squeeze(-1)


print("✅ Model classes loaded")

---
## Load Elevation Data & Sampling Utils

In [ ]:
print("="*70)
print("LOADING ETOPO ELEVATION DATA")
print("="*70)

if 'COLAB_GPU' in os.environ:
    !wget -q -O etopo_60s.nc "https://www.ngdc.noaa.gov/thredds/fileServer/global/ETOPO2022/60s/60s_surface_elev_netcdf/ETOPO_2022_v1_60s_N90W180_surface.nc"
    data_path = 'etopo_60s.nc'
else:
    data_path = 'etopo_60s.nc'

ds = xr.open_dataset(data_path)
elevation = ds['z'].values
lats = ds['lat'].values
lons = ds['lon'].values

print(f"✅ Elevation: {elevation.shape}")
print(f"   Range: [{elevation.min():.2f}, {elevation.max():.2f}] m")
print("="*70)

In [ ]:
# Copy sample_regional_blocked and train_elevation_model from NB21
# (Same functions, using same sampling strategy)

def sample_regional_blocked(data, lons, lats, region_bounds, n_samples=5000,
                           grid_size=5.0, test_ratio=0.3, seed=42):
    """Sample from regional subset with spatial blocking."""
    np.random.seed(seed)
    
    lat_mask = (lats >= region_bounds['lat_min']) & (lats <= region_bounds['lat_max'])
    lon_mask = (lons >= region_bounds['lon_min']) & (lons <= region_bounds['lon_max'])
    
    lat_idx = np.where(lat_mask)[0]
    lon_idx = np.where(lon_mask)[0]
    
    regional_data = data[np.ix_(lat_idx, lon_idx)]
    regional_lats = lats[lat_idx]
    regional_lons = lons[lon_idx]
    
    lon_grid, lat_grid = np.meshgrid(regional_lons, regional_lats)
    valid = regional_data > -1e30
    
    valid_lons = lon_grid[valid]
    valid_lats = lat_grid[valid]
    valid_vals = regional_data[valid]
    
    n_valid = len(valid_vals)
    if n_valid > n_samples:
        sample_idx = np.random.choice(n_valid, n_samples, replace=False)
        sample_lons = valid_lons[sample_idx]
        sample_lats = valid_lats[sample_idx]
        sample_vals = valid_vals[sample_idx]
    else:
        sample_lons = valid_lons
        sample_lats = valid_lats
        sample_vals = valid_vals
    
    region_width = region_bounds['lon_max'] - region_bounds['lon_min']
    region_height = region_bounds['lat_max'] - region_bounds['lat_min']
    
    n_lon_cells = max(1, int(region_width / grid_size))
    n_lat_cells = max(1, int(region_height / grid_size))
    n_cells = n_lon_cells * n_lat_cells
    
    test_cells = set(np.random.choice(n_cells, max(1, int(n_cells * test_ratio)), replace=False))
    
    train_mask = []
    for lon, lat in zip(sample_lons, sample_lats):
        lon_cell = int((lon - region_bounds['lon_min']) / grid_size)
        lat_cell = int((lat - region_bounds['lat_min']) / grid_size)
        lon_cell = min(lon_cell, n_lon_cells - 1)
        lat_cell = min(lat_cell, n_lat_cells - 1)
        cell = lat_cell * n_lon_cells + lon_cell
        train_mask.append(cell not in test_cells)
    
    train_mask = np.array(train_mask)
    coords = np.stack([sample_lons, sample_lats], axis=1)
    
    return coords[train_mask], sample_vals[train_mask], coords[~train_mask], sample_vals[~train_mask]


def train_elevation_model(name, encoder, coords_train, vals_train, coords_test, vals_test,
                         epochs=100, batch_size=256, lr=1e-3):
    """Train elevation prediction model."""
    model = RegressionPredictor(encoder).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    
    # Normalize + shift + log
    train_mean, train_std = vals_train.mean(), vals_train.std()
    vals_train_norm = (vals_train - train_mean) / train_std
    vals_test_norm = (vals_test - train_mean) / train_std
    
    shift = -vals_train_norm.min() + 1
    train_y = np.log1p(vals_train_norm + shift)
    test_y = np.log1p(vals_test_norm + shift)
    
    train_X = torch.tensor(coords_train, dtype=torch.float32)
    train_y = torch.tensor(train_y, dtype=torch.float32)
    test_X = torch.tensor(coords_test, dtype=torch.float32).to(device)
    test_y = torch.tensor(test_y, dtype=torch.float32)
    
    loader = DataLoader(TensorDataset(train_X, train_y),
                       batch_size=batch_size, shuffle=True)
    
    best_r2 = -float('inf')
    start = time.time()
    
    for epoch in range(epochs):
        model.train()
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            opt.zero_grad()
            loss = loss_fn(model(X), y)
            loss.backward()
            opt.step()
        
        if (epoch + 1) % 10 == 0 or epoch == epochs - 1:
            model.eval()
            with torch.no_grad():
                pred = model(test_X).cpu().numpy()
            r2 = r2_score(test_y.numpy(), pred)
            best_r2 = max(best_r2, r2)
    
    train_time = time.time() - start
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    return {
        'model': name,
        'r2': best_r2,
        'params': n_params,
        'time': train_time,
    }


print("✅ Utilities loaded")

---
## Experiment: L=10 vs L=40 Regional Comparison

**Config**: 20K samples, 5 seeds, 2 regions, 2 L-values, 2 activations

In [ ]:
print("="*80)
print("L=10 vs L=40 REGIONAL COMPARISON")
print("="*80)

REGIONS = {
    'asia_himalayas': {
        'lat_min': 25, 'lat_max': 40,
        'lon_min': 70, 'lon_max': 100,
        'terrain': 'mountainous',
    },
    'africa_sahara': {
        'lat_min': 15, 'lat_max': 30,
        'lon_min': -10, 'lon_max': 30,
        'terrain': 'flat',
    },
}

L_VALUES = [10, 40]
ACTIVATIONS = ['relu', 'spline']
SEEDS = range(42, 47)  # 5 seeds
N_SAMPLES = 20000
EPOCHS = 100

results = []

for region_name, region_bounds in REGIONS.items():
    print(f"\n{'='*80}")
    print(f"Region: {region_name.upper()} ({region_bounds['terrain']})")
    print("="*80)
    
    for L in L_VALUES:
        print(f"\n  L={L} ({(L+1)**2} dims):")
        
        for seed in SEEDS:
            print(f"    Seed {seed}: ", end="")
            
            coords_train, vals_train, coords_test, vals_test = sample_regional_blocked(
                elevation, lons, lats, region_bounds, n_samples=N_SAMPLES, seed=seed
            )
            
            for act in ACTIVATIONS:
                kwargs = {'n_knots': 15, 'init': 'relu'} if act == 'spline' else None
                
                enc = UniversalEncoder(
                    input_type='sh',
                    sh_legendre_polys=L,
                    activation_type=act,
                    activation_kwargs=kwargs
                )
                
                res = train_elevation_model(
                    f'{region_name}_L{L}_seed{seed}_{act}',
                    enc,
                    coords_train, vals_train,
                    coords_test, vals_test,
                    epochs=EPOCHS
                )
                
                res['seed'] = seed
                res['activation'] = act
                res['region'] = region_name
                res['terrain'] = region_bounds['terrain']
                res['L'] = L
                res['n_samples'] = N_SAMPLES
                
                results.append(res)
                print(f"{act}={res['r2']:.4f} ", end="")
            
            print()

df = pd.DataFrame(results)

print("\n" + "="*80)
print("EXPERIMENT COMPLETE")
print("="*80)
print(df[['region', 'L', 'seed', 'activation', 'r2', 'params']].to_string(index=False))

---
## Analysis: Does L=40 Help Regionally?

In [ ]:
print("\n" + "="*80)
print("ANALYSIS: L=10 vs L=40 REGIONAL PERFORMANCE")
print("="*80)

for region_name in REGIONS.keys():
    region_data = df[df['region'] == region_name]
    terrain = region_data['terrain'].iloc[0]
    
    print(f"\n{'='*80}")
    print(f"{region_name.upper()} ({terrain})")
    print("="*80)
    
    for act in ACTIVATIONS:
        act_data = region_data[region_data['activation'] == act]
        
        L10_r2s = act_data[act_data['L'] == 10]['r2'].values
        L40_r2s = act_data[act_data['L'] == 40]['r2'].values
        
        L10_mean = L10_r2s.mean()
        L40_mean = L40_r2s.mean()
        
        improvement = 100 * (L40_mean - L10_mean) / L10_mean
        
        # Paired t-test
        if len(L10_r2s) == len(L40_r2s) and len(L10_r2s) > 1:
            t_stat, p_value = stats.ttest_rel(L40_r2s, L10_r2s)
        else:
            t_stat, p_value = np.nan, np.nan
        
        print(f"\n--- {act.upper()} ---")
        print(f"L=10: R² = {L10_mean:.4f} ± {L10_r2s.std():.4f} ({(10+1)**2} dims)")
        print(f"L=40: R² = {L40_mean:.4f} ± {L40_r2s.std():.4f} ({(40+1)**2} dims)")
        print(f"Improvement: {improvement:+.2f}%")
        
        if not np.isnan(p_value):
            print(f"Paired t-test: t={t_stat:.3f}, p={p_value:.4f}")
            if p_value < 0.05:
                if improvement > 0:
                    print("  ✅ SIGNIFICANT: L=40 wins (p < 0.05)")
                else:
                    print("  ⚠️  SIGNIFICANT: L=10 wins (p < 0.05) - unexpected!")
            else:
                print("  ❌ NOT SIGNIFICANT: No clear difference (p ≥ 0.05)")
        
        # Parameter cost-benefit
        L10_params = act_data[act_data['L'] == 10]['params'].iloc[0]
        L40_params = act_data[act_data['L'] == 40]['params'].iloc[0]
        param_increase = L40_params - L10_params
        
        print(f"\nParameter cost: +{param_increase:,} params ({param_increase / L10_params * 100:.1f}% increase)")
        
        if improvement > 1.0 and (p_value < 0.05 or np.isnan(p_value)):
            print("  🎯 WORTH IT: Improvement justifies parameter cost")
        elif improvement > 0:
            print("  ⚠️  MARGINAL: Small improvement, may not justify cost")
        else:
            print("  ❌ NOT WORTH IT: No improvement from L=40")

print("\n" + "="*80)
print("SUMMARY")
print("="*80)

print("\nCompare terrain types:")
for act in ACTIVATIONS:
    himalayas_L10 = df[(df['region'] == 'asia_himalayas') & (df['L'] == 10) & (df['activation'] == act)]['r2'].mean()
    himalayas_L40 = df[(df['region'] == 'asia_himalayas') & (df['L'] == 40) & (df['activation'] == act)]['r2'].mean()
    himalayas_gain = 100 * (himalayas_L40 - himalayas_L10) / himalayas_L10
    
    sahara_L10 = df[(df['region'] == 'africa_sahara') & (df['L'] == 10) & (df['activation'] == act)]['r2'].mean()
    sahara_L40 = df[(df['region'] == 'africa_sahara') & (df['L'] == 40) & (df['activation'] == act)]['r2'].mean()
    sahara_gain = 100 * (sahara_L40 - sahara_L10) / sahara_L10
    
    print(f"\n{act.upper()}:")
    print(f"  Himalayas (complex): L=40 gain = {himalayas_gain:+.2f}%")
    print(f"  Sahara (flat):       L=40 gain = {sahara_gain:+.2f}%")
    
    if himalayas_gain > sahara_gain + 1.0:
        print("  → L=40 helps more for complex terrain (as expected)")
    elif abs(himalayas_gain - sahara_gain) < 1.0:
        print("  → L=40 effect similar across terrains")
    else:
        print("  → Unexpected pattern")

print("\n" + "="*80)
print("RECOMMENDATION")
print("="*80)

print("\nBased on results:")
print("IF L=40 shows >3% improvement on complex terrain:")
print("  → Use L=40 for regional tasks")
print("  → Justifies 13× parameter increase")
print("\nIF L=40 shows 1-3% improvement:")
print("  → Marginal benefit, consider task requirements")
print("\nIF L=40 shows <1% improvement:")
print("  → Stick with L=10, save parameters")
print("="*80)

---
## Optional: Save Results

In [ ]:
if 'COLAB_GPU' in os.environ:
    save_dir = '/content/drive/MyDrive/learned_activation_results/nb21d/'
    os.makedirs(save_dir, exist_ok=True)
    
    df.to_csv(f'{save_dir}L10_vs_L40_regional_comparison.csv', index=False)
    
    print(f"✅ Results saved to: {save_dir}")
else:
    print("Not on Colab, skip Drive save")